# 05 — Ablation Studies: What Makes Strategy Recognition Work?

## Goal

Our system recognises badminton tactical strategies (e.g. *attack*, *defend*, *move to net*) from skeleton pose sequences using a few-shot learning pipeline: **skeleton features → encoder → Prototypical Network**.

There are many design choices in this pipeline. **Ablation studies isolate each choice and measure its impact**, so we know *which decisions actually matter* rather than guessing.

## Method — Sequential "Fix the Best, Vary One"

We run ablations **one axis at a time**. After each step, the winning setting is locked in and carried forward:

| Step | Question | What we vary | What's fixed |
|------|----------|-------------|-------------|
| **1 — Features** | *Which input signals carry strategy-relevant information?* | Feature layers L0–L3 (raw xy → kinematics → court context → joint angles) | ST-GCN encoder, dual-player graph, ProtoNet |
| **2 — Encoder** | *Does the model architecture matter?* | ST-GCN vs Transformer | Best features from Step 1 |
| **3 — Pre-training** | *Does SSL pre-training on ShuttleSet help?* (RQ2) | Random init vs SimCLR vs SupCon | Best features + encoder |
| **4 — Classifier** | *Is ProtoNet the best few-shot method?* | ProtoNet vs k-NN vs Linear probe | Best features + encoder + pre-training |
| **5 — K-Shot** | *How many examples per class do we need?* | K = 1, 3, 5, 8, 10 | All best settings |
| **6 — Shuttle** | *Does adding shuttlecock trajectory help?* | Skeleton-only (34 nodes) vs +shuttle (35 nodes) | All best settings |

## Evaluation

All results use **5-fold stratified cross-validation** on the 30 training rallies (10 held-out rallies are never touched here). The primary metric is **Macro-F1** — the unweighted average of per-class F1 scores — which treats all 5 strategy classes equally regardless of how frequent they are. A random baseline would score ~0.20.

In [ ]:
import os, sys, zipfile
from pathlib import Path

# ── Colab detection ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'

    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print(f'Extracted to {PROJECT_PATH}')
    else:
        print('Project already extracted.')

    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    # Override paths so outputs go to Drive (not ephemeral /content)
    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.FB_SKELETONS_GDINO_V2 = DRIVE_ROOT / 'datasets_preprocessing' / 'finebadminton_skeletons_gdino_v2'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'Datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    # ── ShuttleSet CSV annotations ────────────────────────────────────────
    _cfg.SS_CSV_ROOT  = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV = _cfg.SS_CSV_ROOT / 'match.csv'
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'

    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f'Drive root   : {DRIVE_ROOT}')
    print(f'Models dir   : {_cfg.MODELS_DIR}')
    print(f'SS skeletons : {_cfg.SS_SKELETONS_GDINO}')
    print(f'FB skeletons : {_cfg.FB_SKELETONS_GDINO_V2}')
    print(f'SS CSV root  : {_cfg.SS_CSV_ROOT}')
    print(f'SS CSV exists: {_cfg.SS_CSV_ROOT.exists()}')
else:
    sys.path.insert(0, os.path.abspath('..'))
    DRIVE_ROOT = Path('..')
    print('Local run — using paths from config.py')

import torch
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm
from src.config import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'MODELS_DIR: {MODELS_DIR}')

SKELETON_DIR = FB_SKELETONS_GDINO_V2
print(f'Skeleton dir: {SKELETON_DIR}')


## Ablation Runner — Shared Infrastructure

The `run_ablation()` function below is the workhorse for every step. Given a config, it:

1. **Builds the graph** (34-node dual-player skeleton, or 17-node single-player, or 35-node with shuttle)
2. **Loads the dataset** with the specified feature layer
3. **Splits into 5 folds** (stratified by strategy label, rally-level — no data leakage across rallies)
4. **Per fold:** trains the encoder with episodic ProtoNet for 30 epochs (50 episodes each), picks the best checkpoint by validation F1, then evaluates on the held-out fold
5. **Returns** mean and std of Macro-F1 across folds, plus per-class breakdown

Each step prints fold-level results as it goes.

In [ ]:
from src.data.graph_builder import GraphBuilder
from src.data.dataset import FineBadmintonDataset, EpisodicSampler, collate_episode
from src.models.stgcn_model import STGCN
from src.models.transformer_encoder import SkeletonTransformer
from src.models.proto_net import PrototypicalNetwork
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# ── Speed overrides (halves total runtime) ────────────────────────────────
ABLATION_EPOCHS = 30              # default 50 → 30 (most learning is early)
ABLATION_EPISODES_PER_EPOCH = 50  # default 100 → 50


def build_encoder(cfg, adjacency, device):
    """Build encoder; derives num_nodes from adjacency shape to handle single_player."""
    feature_dim = FEATURE_DIMS[cfg.ablation.feature_layer]
    enc_type = cfg.ablation.encoder
    num_nodes = adjacency.shape[-1]  # 17, 34, or 35 depending on single_player/use_shuttle

    if enc_type == 'stgcn':
        return STGCN(
            in_channels=feature_dim,
            num_nodes=num_nodes,
            adjacency=adjacency,
            num_layers=cfg.stgcn.num_layers,
            base_channels=cfg.stgcn.base_channels,
            embedding_dim=cfg.stgcn.embedding_dim,
            temporal_kernel=cfg.stgcn.temporal_kernel,
            dropout=cfg.stgcn.dropout,
        ).to(device)
    elif enc_type == 'transformer':
        return SkeletonTransformer(
            in_channels=feature_dim * num_nodes,  # derived, not hardcoded
            d_model=cfg.transformer.d_model,
            nhead=cfg.transformer.nhead,
            num_layers=cfg.transformer.num_layers,
            embedding_dim=cfg.transformer.embedding_dim,
            dropout=cfg.transformer.dropout,
        ).to(device)
    else:
        raise ValueError(f"Unknown encoder: {enc_type}. LSTM/CNN1D are stretch goals — not in scope.")


def extract_embeddings(encoder, dataset, indices, device):
    encoder.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for i in indices:
            x, y = dataset[i]
            x = x.unsqueeze(0).to(device)
            emb = encoder(x).cpu().numpy()[0]
            embeddings.append(emb)
            labels.append(y)
    return np.array(embeddings), np.array(labels)


def train_one_epoch(encoder, dataset, train_idx, optimizer, cfg, device):
    encoder.train()
    train_labels = [dataset.get_labels()[i] for i in train_idx]
    sampler = EpisodicSampler(
        labels=train_labels,
        n_way=cfg.proto.n_way,
        k_shot=cfg.proto.k_shot,
        n_query=cfg.proto.n_query,
        episodes_per_epoch=ABLATION_EPISODES_PER_EPOCH,
    )
    epoch_loss, epoch_acc, n_episodes = 0.0, 0.0, 0
    proto_net = PrototypicalNetwork(encoder, distance=cfg.proto.distance)

    for episode_indices in sampler:
        actual_indices = [train_idx[i] for i in episode_indices]
        batch = [dataset[i] for i in actual_indices]
        support_x, support_y, query_x, query_y = collate_episode(
            batch, cfg.proto.n_way, cfg.proto.k_shot, cfg.proto.n_query
        )
        support_x, support_y = support_x.to(device), support_y.to(device)
        query_x, query_y = query_x.to(device), query_y.to(device)

        support_emb = encoder(support_x)
        query_emb = encoder(query_x)
        n_way_actual = int(support_y.unique().numel())
        prototypes = proto_net.compute_prototypes(support_emb, support_y, n_way_actual)
        distances = proto_net.compute_distances(query_emb, prototypes)
        log_probs = torch.nn.functional.log_softmax(-distances, dim=1)
        loss = torch.nn.functional.nll_loss(log_probs, query_y)
        acc = ((-distances).argmax(dim=1) == query_y).float().mean().item()

        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc
        n_episodes += 1

    return epoch_loss / n_episodes, epoch_acc / n_episodes


def evaluate_protonet(encoder, dataset, support_idx, test_idx, device):
    support_emb, support_labels = extract_embeddings(encoder, dataset, support_idx, device)
    test_emb, test_labels = extract_embeddings(encoder, dataset, test_idx, device)

    prototypes = {c: support_emb[support_labels == c].mean(axis=0)
                  for c in np.unique(support_labels)}
    preds = np.array([
        min(prototypes, key=lambda c: np.linalg.norm(emb - prototypes[c]))
        for emb in test_emb
    ])
    all_labels = list(range(NUM_CLASSES))
    macro_f1 = f1_score(test_labels, preds, average='macro',
                        labels=all_labels, zero_division=0)
    per_class = f1_score(test_labels, preds, average=None,
                         labels=all_labels, zero_division=0)
    return {'macro_f1': macro_f1, 'per_class_f1': per_class.tolist()}


def run_ablation(cfg, skeleton_dir=None, ssl_path=None, device=device):
    """Full ablation run with 5-fold CV on rally-level train split. Returns aggregated results dict."""
    if skeleton_dir is None:
        skeleton_dir = SKELETON_DIR

    graph_builder = GraphBuilder(
        use_inter_player=cfg.ablation.use_inter_player,
        single_player=cfg.ablation.single_player,
    )
    adjacency = graph_builder.build_adjacency().to(device)
    num_nodes = adjacency.shape[-1]  # 17 or 34

    # Single-player transform: slice dataset output to first 17 nodes
    transform = None
    if cfg.ablation.single_player:
        transform = lambda x: x[:, :, :num_nodes]

    dataset = FineBadmintonDataset(
        skeleton_dir=skeleton_dir,
        shot_window=cfg.data.shot_window,
        feature_layer=cfg.ablation.feature_layer,
        transform=transform,
    )

    # Rally-level splits: 30 train / 10 held-out
    train_rally_idx, _ = dataset.get_rally_splits(n_train_rallies=30, seed=cfg.data.random_seed)
    train_labels = [dataset.get_labels()[i] for i in train_rally_idx]

    # 5-fold CV on training rallies only
    skf = StratifiedKFold(n_splits=cfg.data.num_folds, shuffle=True, random_state=cfg.data.random_seed)
    splits = []
    for tv_idx, test_idx in skf.split(train_rally_idx, train_labels):
        tv_labels = [train_labels[i] for i in tv_idx]
        inner_skf = StratifiedKFold(n_splits=max(2, cfg.data.num_folds - 1),
                                     shuffle=True, random_state=cfg.data.random_seed)
        tr_idx, val_idx = next(inner_skf.split(tv_idx, tv_labels))
        splits.append((
            [train_rally_idx[i] for i in tv_idx[tr_idx]],
            [train_rally_idx[i] for i in tv_idx[val_idx]],
            [train_rally_idx[i] for i in test_idx],
        ))

    # Load SSL weights if available
    ssl_checkpoint = None
    if ssl_path is None:
        # Try supcon first, then simclr, then legacy naming
        layer = cfg.ablation.feature_layer
        for method in ('supcon', 'simclr'):
            candidate = MODELS_DIR / f'ssl_pretrained_{method}_{layer}.pt'
            if candidate.exists():
                ssl_path = candidate
                break
        else:
            ssl_path = MODELS_DIR / f'ssl_pretrained_{layer}.pt'  # legacy
    if cfg.ablation.use_pretrained and ssl_path.exists() and cfg.ablation.encoder == 'stgcn':
        ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
        print(f"  SSL weights: {ssl_path.name}")
    else:
        print(f"  No SSL weights — random init")

    fold_f1s, fold_per_class = [], []

    for fold_idx, (tr_idx, val_idx, te_idx) in enumerate(splits):
        encoder = build_encoder(cfg, adjacency, device)
        if ssl_checkpoint is not None:
            encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

        optimizer = optim.Adam(
            encoder.parameters(),
            lr=cfg.proto.lr * cfg.proto.encoder_lr_scale,
            weight_decay=1e-5,
        ) if cfg.proto.fine_tune_encoder else None

        if optimizer is None:
            encoder.eval()
            for p in encoder.parameters():
                p.requires_grad = False

        best_val_f1, best_state = 0.0, None
        for epoch in range(ABLATION_EPOCHS):
            train_one_epoch(encoder, dataset, tr_idx, optimizer, cfg, device)
            if (epoch + 1) % 5 == 0:
                val_res = evaluate_protonet(encoder, dataset, tr_idx, val_idx, device)
                if val_res['macro_f1'] > best_val_f1:
                    best_val_f1 = val_res['macro_f1']
                    best_state = {k: v.clone() for k, v in encoder.state_dict().items()}

        if best_state is not None:
            encoder.load_state_dict(best_state)

        test_result = evaluate_protonet(encoder, dataset, tr_idx, te_idx, device)
        fold_f1s.append(test_result['macro_f1'])
        fold_per_class.append(test_result['per_class_f1'])
        print(f"    Fold {fold_idx+1}: F1={test_result['macro_f1']:.3f}")

    result = {
        'macro_f1_mean': float(np.mean(fold_f1s)),
        'macro_f1_std': float(np.std(fold_f1s)),
        'fold_f1s': [float(f) for f in fold_f1s],
        'per_class_f1_mean': np.mean(fold_per_class, axis=0).tolist(),
    }
    print(f"  → Macro-F1: {result['macro_f1_mean']:.3f} ± {result['macro_f1_std']:.3f}")
    return result

print(f"Ablation runner ready. Speed: {ABLATION_EPOCHS} epochs × {ABLATION_EPISODES_PER_EPOCH} episodes/epoch")

---
## Step 1 — Feature Engineering: Which Input Signals Matter?

**Why this matters:** Before the model even sees the data, we decide *what information to compute from the raw skeleton*. This is often the single biggest factor in performance — a model can only learn from the signals it receives.

We test four progressively richer feature layers. Each layer **adds** features on top of the previous one:

| Layer | Features added | Total dim | Intuition |
|-------|---------------|----------:|-----------|
| **L0** | Raw x, y joint positions | 2 | Baseline — just "where are the joints?" |
| **L1** | + velocity, acceleration | 6 | Captures *how fast* players move — are they lunging, recovering, stationary? |
| **L2** | + distance to net, court centre, opponent | 9 | Court-awareness — is the player at the net or baseline? How far from the opponent? |
| **L3** | + elbow, shoulder, knee angles | 12 | Body pose details — racket preparation, stance, reach |

**What we expect:** L0 alone may struggle because absolute pixel positions vary across camera angles. L1 adds motion dynamics. L2 adds spatial strategy context (court positioning is key to badminton tactics). L3 tests whether fine-grained body angles provide additional value or just add noise.

**Fixed settings:** ST-GCN encoder, full dual-player graph (34 nodes), episodic ProtoNet fine-tuning.
SSL pre-trained weights are loaded for L2 only (only checkpoint available); others use random init.

In [ ]:
step1a_configs = {
    'L0_raw_xy':       get_config('L0', **{'ablation.feature_layer': 'L0', 'ablation.use_pretrained': False}),
    'L1_kinematics':   get_config('L1', **{'ablation.feature_layer': 'L1', 'ablation.use_pretrained': False}),
    'L2_court_ctx':    get_config('L2', **{'ablation.feature_layer': 'L2'}),  # SSL checkpoint exists
    'L3_joint_angles': get_config('L3', **{'ablation.feature_layer': 'L3', 'ablation.use_pretrained': False}),
}

print(f"Step 1a — Feature Engineering ({len(step1a_configs)} variants)")
print(f"{'Variant':<20} {'Layer':<5} {'Dim':>4}  {'Init'}")
print("-" * 45)
for name, cfg in step1a_configs.items():
    fl = cfg.ablation.feature_layer
    ssl = 'SSL' if cfg.ablation.use_pretrained else 'random'
    print(f"  {name:<20} {fl:<5} {FEATURE_DIMS[fl]:>4}  {ssl}")

In [ ]:
step1a_results = {}

for name, cfg in step1a_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 1a — {name} ({cfg.ablation.feature_layer}, {FEATURE_DIMS[cfg.ablation.feature_layer]}-dim)")
    step1a_results[name] = run_ablation(cfg, device=device)

# Summary table
print(f"\n{'='*60}")
print(f"Step 1a Summary — Feature Engineering")
print(f"{'Variant':<22} {'Macro-F1':>10} {'± Std':>8}  Per-class F1")
print("-" * 70)
best_name_1a, best_f1_1a = None, 0
for name, res in step1a_results.items():
    pc = [f"{v:.2f}" for v in res['per_class_f1_mean']]
    print(f"  {name:<20} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}  {pc}")
    if res['macro_f1_mean'] > best_f1_1a:
        best_f1_1a = res['macro_f1_mean']
        best_name_1a = name

print(f"\nBest: {best_name_1a} ({best_f1_1a:.3f})")
BEST_FEATURE_LAYER = step1a_configs[best_name_1a].ablation.feature_layer
BEST_STEP1 = BEST_FEATURE_LAYER  # alias used by later steps
print(f"→ Fixing feature_layer = {BEST_FEATURE_LAYER} for Steps 2–6")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
names = list(step1a_results.keys())
means = [step1a_results[n]['macro_f1_mean'] for n in names]
stds  = [step1a_results[n]['macro_f1_std']  for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=5,
              color=['#4C72B0','#55A868','#C44E52','#8172B2'])
ax.set_ylabel('Macro-F1')
ax.set_title('Step 1a — Feature Engineering Ablation')
ax.set_ylim(0, 0.8)
ax.axhline(0.2, color='gray', linestyle='--', linewidth=0.8, label='Random baseline (5-class)')
ax.legend()
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_feature_layers.png', dpi=150)
plt.show()

# Step 1 results feed into the combined step1_results for final save
step1_results = step1a_results

---
## Step 2 — Encoder Architecture: Does Graph Structure Help?

**Why this matters:** The encoder transforms skeleton sequences into fixed-size embeddings. We compare two fundamentally different approaches:

| Encoder | How it works | Strength |
|---------|-------------|----------|
| **ST-GCN** | Graph convolutions that respect the skeleton topology — information flows along bones and across time | Has a *structural prior*: it knows joints are connected by bones, so it can learn local motion patterns (e.g. arm swing) efficiently |
| **Transformer** | Self-attention over all joints at all timesteps — every joint can attend to every other | Fully flexible — can discover long-range dependencies (e.g. footwork predicting racket motion) but has no built-in skeleton structure |

**What we expect:** ST-GCN should have an advantage on our small dataset because its graph structure acts as a regulariser. The Transformer is more expressive but needs more data to learn what ST-GCN gets "for free" from the skeleton topology.

**Fixed settings:** Best feature layer from Step 1, dual-player graph, ProtoNet.
Note: SSL pre-trained weights only exist for ST-GCN, so the Transformer uses random init.

In [ ]:
step2_configs = {
    'stgcn':       get_config('stgcn', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'stgcn',
    }),
    'transformer': get_config('transformer', **{
        'ablation.feature_layer': BEST_FEATURE_LAYER,
        'ablation.encoder': 'transformer',
        'ablation.use_pretrained': False,  # No SSL weights for Transformer
    }),
}

print(f"Step 2 — Encoder Architecture")
print(f"  Fixed: feature={BEST_FEATURE_LAYER}")
for name in step2_configs:
    print(f"  {name}")

In [ ]:
step2_results = {}

for name, cfg in step2_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 2 — {name}")
    step2_results[name] = run_ablation(cfg, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 2 Summary — Encoder Architecture")
print(f"  Fixed: feature={BEST_FEATURE_LAYER}")
print(f"{'Encoder':<15} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 35)
best_name_2, best_f1_2 = None, 0
for name, res in step2_results.items():
    print(f"  {name:<13} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")
    if res['macro_f1_mean'] > best_f1_2:
        best_f1_2 = res['macro_f1_mean']
        best_name_2 = name

print(f"\nBest: {best_name_2} ({best_f1_2:.3f})")
BEST_ENCODER = best_name_2
BEST_STEP2 = BEST_ENCODER  # alias used by later steps
print(f"→ Fixing encoder = {BEST_ENCODER} for Steps 3–6")

---
## Step 3 — Pre-Training Regime: Does SSL on ShuttleSet Transfer? (RQ2)

**Why this matters:** We have very few labelled FineBadminton samples (~355 shots across 40 rallies). Self-supervised pre-training on the larger, unlabelled ShuttleSet (~26 matches) could give the encoder a better starting point — learning general skeleton motion patterns before seeing any strategy labels.

We compare three initialisation strategies:

| Variant | What it does | Intuition |
|---------|-------------|-----------|
| **random_init** | Encoder starts from scratch | Baseline — learns everything from the small labelled set |
| **SimCLR** | Pre-trained with NT-Xent contrastive loss (no labels) | Learns that augmented versions of the same skeleton sequence should map nearby in embedding space. Purely self-supervised — doesn't use any shot-type or strategy labels |
| **SupCon** | Pre-trained with supervised contrastive loss using ShuttleSet shot-type labels | Same idea, but pulls together sequences with the *same shot type* (e.g. all smashes). Uses ShuttleSet's shot labels as a proxy for strategy |

**What we expect:** SSL should help because it gives the encoder exposure to diverse badminton movement patterns. SupCon may outperform SimCLR since shot types partially correlate with strategy.

**Fixed settings:** Best feature layer + encoder from Steps 1–2.
Checkpoints at `models/ssl_pretrained_{method}_{layer}.pt` — missing ones are skipped automatically.

In [ ]:
step3_configs = {
    'random_init': get_config('random', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': False,
    }),
}

# Add SSL variants if checkpoints exist
ssl_checkpoint_paths = {}
for method in ('simclr', 'supcon'):
    ckpt = MODELS_DIR / f'ssl_pretrained_{method}_{BEST_STEP1}.pt'
    if ckpt.exists():
        step3_configs[method] = get_config(method, **{
            'ablation.feature_layer': BEST_STEP1,
            'ablation.encoder': BEST_STEP2,
            'ablation.use_pretrained': True,
        })
        ssl_checkpoint_paths[method] = ckpt
        print(f"Found checkpoint: {ckpt.name}")
    else:
        print(f"No checkpoint for {method} — skipping")

print(f"\nStep 3 variants (feature={BEST_STEP1}, encoder={BEST_STEP2}):")
for name in step3_configs:
    print(f"  {name}")

In [ ]:
step3_results = {}

for name, cfg in step3_configs.items():
    print(f"\n{'='*50}")
    print(f"Step 3 — {name}")
    ssl_path = ssl_checkpoint_paths.get(name)  # None for random_init
    step3_results[name] = run_ablation(cfg, ssl_path=ssl_path, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 3 Summary — Pre-Training Regime (RQ2)")
print(f"{'Variant':<25} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 45)
best_name, best_f1 = None, 0
for name, res in step3_results.items():
    print(f"{name:<25} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")
    if res['macro_f1_mean'] > best_f1:
        best_f1 = res['macro_f1_mean']
        best_name = name
print(f"\nBest: {best_name} ({best_f1:.3f})")

BEST_STEP3_PRETRAINED = step3_configs[best_name].ablation.use_pretrained
BEST_STEP3_METHOD = best_name  # 'random_init' | 'simclr' | 'supcon'
print(f"→ Fixing use_pretrained = {BEST_STEP3_PRETRAINED}, method = {BEST_STEP3_METHOD} for Step 4")

---
## Step 4 — Few-Shot Classifier: How Should We Use the Embeddings?

**Why this matters:** The encoder produces a 128-dim embedding for each skeleton sequence. But how should we classify a new, unseen shot? We train the encoder once (with episodic ProtoNet), freeze it, then compare different classifiers on top of the same embeddings.

| Method | How it classifies | Strengths |
|--------|------------------|-----------|
| **ProtoNet** | Compute class centroids (prototypes) from support set, assign query to nearest prototype | Designed for few-shot; no extra parameters to fit |
| **k-NN (k=3, 5)** | Find k nearest training embeddings, majority vote | Simple, non-parametric; no assumptions about class shape |
| **Linear probe** | Logistic regression on frozen embeddings | Tests whether classes are linearly separable in embedding space |

**What we expect:** If the encoder has learned good representations, all classifiers should perform similarly. If ProtoNet significantly outperforms linear probe, it suggests the embedding space has non-linear class boundaries.

**Fixed settings:** Best feature layer + encoder + pre-training from Steps 1–3.

In [ ]:
step4_methods = {
    'protonet': {'method': 'protonet'},
    'knn_3': {'method': 'knn', 'knn_k': 3},
    'knn_5': {'method': 'knn', 'knn_k': 5},
    'linear_probe': {'method': 'linear_probe'},
}

print(f"Step 4 — Classifier comparison")
print(f"Using best config: feature={BEST_STEP1}, encoder={BEST_STEP2}, pretrained={BEST_STEP3_PRETRAINED}")
for name in step4_methods:
    print(f"  {name}")

In [ ]:
# Train the best encoder once, extract embeddings, then compare classifiers
best_cfg = get_config('step4_best', **{
    'ablation.feature_layer': BEST_STEP1,
    'ablation.encoder': BEST_STEP2,
    'ablation.use_pretrained': BEST_STEP3_PRETRAINED,
})

# Build graph and dataset
graph_builder = GraphBuilder(
    use_inter_player=best_cfg.ablation.use_inter_player,
    single_player=best_cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)

dataset = FineBadmintonDataset(
    skeleton_dir=SKELETON_DIR,
    shot_window=best_cfg.data.shot_window,
    feature_layer=BEST_STEP1,
)
splits = dataset.get_fold_splits(n_folds=best_cfg.data.num_folds, seed=best_cfg.data.random_seed)

# Load SSL if available — try method-specific checkpoint first
ssl_checkpoint = None
if BEST_STEP3_PRETRAINED and BEST_STEP2 == 'stgcn':
    for method in (BEST_STEP3_METHOD, 'supcon', 'simclr'):
        ssl_path = MODELS_DIR / f'ssl_pretrained_{method}_{BEST_STEP1}.pt'
        if ssl_path.exists():
            ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
            print(f"SSL weights: {ssl_path.name}")
            break
    if ssl_checkpoint is None:
        # Legacy fallback
        ssl_path = MODELS_DIR / f'ssl_pretrained_{BEST_STEP1}.pt'
        if ssl_path.exists():
            ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)
            print(f"SSL weights (legacy): {ssl_path.name}")

# Train encoder per fold, extract embeddings, evaluate all classifiers
step4_results = {name: [] for name in step4_methods}

for fold_idx, (train_idx, val_idx, test_idx) in enumerate(splits):
    encoder = build_encoder(best_cfg, adjacency, device)
    if ssl_checkpoint is not None:
        encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])

    optimizer = optim.Adam(
        encoder.parameters(),
        lr=best_cfg.proto.lr * best_cfg.proto.encoder_lr_scale,
        weight_decay=1e-5,
    )

    # Episodic training
    best_val_f1, best_state = 0.0, None
    for epoch in range(ABLATION_EPOCHS):
        train_one_epoch(encoder, dataset, train_idx, optimizer, best_cfg, device)
        if (epoch + 1) % 5 == 0:
            val_res = evaluate_protonet(encoder, dataset, train_idx, val_idx, device)
            if val_res['macro_f1'] > best_val_f1:
                best_val_f1 = val_res['macro_f1']
                best_state = {k: v.clone() for k, v in encoder.state_dict().items()}

    if best_state is not None:
        encoder.load_state_dict(best_state)

    # Extract embeddings with frozen encoder
    all_emb, all_labels = extract_embeddings(
        encoder, dataset, list(range(len(dataset))), device
    )

    # Evaluate each classifier
    for name, method_cfg in step4_methods.items():
        X_train, y_train = all_emb[train_idx], all_labels[train_idx]
        X_test, y_test = all_emb[test_idx], all_labels[test_idx]
        all_labels_list = list(range(NUM_CLASSES))

        if method_cfg['method'] == 'protonet':
            prototypes = {}
            for c in np.unique(y_train):
                prototypes[c] = X_train[y_train == c].mean(axis=0)
            preds = np.array([
                min(prototypes, key=lambda c: np.linalg.norm(x - prototypes[c]))
                for x in X_test
            ])
        elif method_cfg['method'] == 'knn':
            clf = KNeighborsClassifier(n_neighbors=method_cfg['knn_k'])
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)
        elif method_cfg['method'] == 'linear_probe':
            clf = LogisticRegression(max_iter=1000, C=1.0)
            clf.fit(X_train, y_train)
            preds = clf.predict(X_test)

        f1 = f1_score(y_test, preds, average='macro',
                       labels=all_labels_list, zero_division=0)
        step4_results[name].append(f1)

    print(f"  Fold {fold_idx+1} done")

# Summary
print(f"\n{'='*60}")
print(f"Step 4 Summary — Few-Shot Method")
print(f"{'Method':<15} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 35)
best_name, best_f1 = None, 0
for name, f1s in step4_results.items():
    mean_f1, std_f1 = np.mean(f1s), np.std(f1s)
    print(f"{name:<15} {mean_f1:>10.3f} {std_f1:>8.3f}")
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_name = name

# Convert to standard format
step4_results = {
    name: {
        'macro_f1_mean': float(np.mean(f1s)),
        'macro_f1_std': float(np.std(f1s)),
        'fold_f1s': [float(f) for f in f1s],
    }
    for name, f1s in step4_results.items()
}

BEST_STEP4 = best_name
print(f"\nBest: {BEST_STEP4} ({best_f1:.3f})")

---
## Step 5 — K-Shot Sensitivity: How Many Examples Do We Need?

**Why this matters:** In few-shot learning, K is the number of labelled examples per class in the support set. This directly answers a practical question: *how much annotation effort is needed?*

We sweep K = 1, 3, 5, 8, 10 while keeping all other settings at their best values from Steps 1–4.

- **K=1** (one-shot): Can the system recognise a strategy from a single example? This is the hardest setting but the most useful if annotation is expensive.
- **K=5** (default): Our standard setting — a reasonable balance between annotation cost and performance.
- **K=10**: Approaching the maximum for our rarest class (*move_to_net* has ~11 samples per fold). Tests whether more examples keep helping.

**Note:** `n_query` is capped so that `K + n_query` doesn't exceed the minimum class count in any fold, preventing episodes from failing to sample enough examples.

**Fixed settings:** All best from Steps 1–4.

In [ ]:
k_values = [1, 3, 5, 8, 10]
step5_results = {}

for k in k_values:
    # Adjust n_query so k + n_query ≤ 11 (move_to_net min in train fold)
    n_query = min(5, max(1, 11 - k))
    print(f"\n{'='*50}")
    print(f"Step 5 — K={k} (n_query={n_query})")

    cfg = get_config(f'kshot_{k}', **{
        'ablation.feature_layer': BEST_STEP1,
        'ablation.encoder': BEST_STEP2,
        'ablation.use_pretrained': BEST_STEP3_PRETRAINED,
        'proto.k_shot': k,
        'proto.n_query': n_query,
    })
    step5_results[k] = run_ablation(cfg, device=device)

# Summary
print(f"\n{'='*60}")
print(f"Step 5 Summary — K-Shot Sensitivity")
print(f"{'K':<5} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 25)
for k, res in step5_results.items():
    print(f"{k:<5} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ks = list(step5_results.keys())
means = [step5_results[k]['macro_f1_mean'] for k in ks]
stds = [step5_results[k]['macro_f1_std'] for k in ks]
ax.errorbar(ks, means, yerr=stds, marker='o', capsize=5, linewidth=2)
ax.set_xlabel('K (support shots per class)')
ax.set_ylabel('Macro-F1')
ax.set_title('K-Shot Sensitivity')
ax.set_xticks(ks)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_kshot_curve.png', dpi=150)
plt.show()

---
## Save Results (Steps 1–5)

Export all ablation results to `results/ablation_results.json` for use in the analysis notebook and report.

In [ ]:
all_results = {
    'best_config': {
        'feature_layer': BEST_STEP1,
        'encoder': BEST_STEP2,
        'use_pretrained': BEST_STEP3_PRETRAINED,
        'classifier': BEST_STEP4,
    },
    'step1_input': step1_results,
    'step2_encoder': step2_results,
    'step3_pretraining': step3_results,
    'step4_classifier': step4_results,
    'step5_kshot': {str(k): v for k, v in step5_results.items()},
    'step6_shuttle': step6_results if 'step6_results' in dir() else {},
}

with open(RESULTS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"Results saved to {RESULTS_DIR / 'ablation_results.json'}")

# Print grand summary
print(f"\n{'='*60}")
print(f"ABLATION SUMMARY")
print(f"{'='*60}")
print(f"Best feature layer:  {BEST_STEP1}")
print(f"Best encoder:        {BEST_STEP2}")
print(f"Best pre-training:   {'SSL+Aux' if BEST_STEP3_PRETRAINED else 'Random init'}")
print(f"Best classifier:     {BEST_STEP4}")
if 'step6_results' in dir():
    best_shuttle = max(step6_results, key=lambda k: step6_results[k]['macro_f1_mean'])
    print(f"Shuttle ablation:    {best_shuttle} wins")
print(f"{'='*60}")


In [ ]:
from src.data.dataset import FineBadmintonDataset

# Check how many rallies have shuttle data
shuttle_dir = FB_SHUTTLES
available_shuttles = list(shuttle_dir.glob("*.npy")) if shuttle_dir.exists() else []
print(f"Shuttle files available: {len(available_shuttles)}")
if available_shuttles:
    ex = np.load(available_shuttles[0])
    print(f"  Shape example: {ex.shape} — (T, 3) [x, y, vis]")
    print(f"  Visibility > 0.5: {(ex[:, 2] > 0.5).sum()}/{len(ex)} frames ({100*(ex[:,2]>0.5).mean():.0f}%)")
else:
    print("  [WARN] No shuttle files found — skeleton+shuttle will use zero-padded shuttle node")

print()

def run_shuttle_ablation(use_shuttle, best_feature_layer, best_encoder_name,
                          best_use_pretrained, cfg_base, device):
    """Run 5-fold CV with or without shuttle node."""
    from src.data.graph_builder import GraphBuilder
    from src.models.proto_net import PrototypicalNetwork
    import torch.optim as optim

    graph_builder = GraphBuilder(
        use_inter_player=cfg_base.ablation.use_inter_player,
        single_player=False,
        use_shuttle=use_shuttle,
    )
    adjacency = graph_builder.build_adjacency().to(device)
    num_nodes = adjacency.shape[-1]  # 34 or 35
    print(f"  Graph: {num_nodes} nodes, adjacency {adjacency.shape}")

    dataset = FineBadmintonDataset(
        skeleton_dir=SKELETON_DIR,
        shot_window=cfg_base.data.shot_window,
        feature_layer=best_feature_layer,
        use_shuttle=use_shuttle,
    )

    # Rally-level splits
    train_rally_idx, holdout_rally_idx = dataset.get_rally_splits(n_train_rallies=30, seed=42)
    train_labels = [dataset.get_labels()[i] for i in train_rally_idx]

    # 5-fold CV on training rallies
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits = []
    for tv_idx, test_idx in skf.split(train_rally_idx, train_labels):
        tv_labels = [train_labels[i] for i in tv_idx]
        inner_skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
        tr_idx, val_idx = next(inner_skf.split(tv_idx, tv_labels))
        splits.append((
            [train_rally_idx[i] for i in tv_idx[tr_idx]],
            [train_rally_idx[i] for i in tv_idx[val_idx]],
            [train_rally_idx[i] for i in test_idx],
        ))

    # Load SSL checkpoint if available
    ssl_path = MODELS_DIR / f'ssl_pretrained_{best_feature_layer}.pt'
    ssl_checkpoint = None
    if best_use_pretrained and ssl_path.exists() and best_encoder_name == 'stgcn':
        ssl_checkpoint = torch.load(ssl_path, map_location=device, weights_only=False)

    fold_f1s = []
    for fold_idx, (tr_idx, val_idx, te_idx) in enumerate(splits):
        encoder = build_encoder(cfg_base, adjacency, device)
        if ssl_checkpoint is not None and not use_shuttle:
            # SSL weights trained on 34-node graph — only load for skeleton-only
            try:
                encoder.load_state_dict(ssl_checkpoint['encoder_state_dict'])
            except RuntimeError:
                pass  # shape mismatch if shuttle changes input dim

        optimizer = optim.Adam(
            encoder.parameters(),
            lr=cfg_base.proto.lr * cfg_base.proto.encoder_lr_scale,
            weight_decay=1e-5,
        )

        best_val_f1, best_state = 0.0, None
        for epoch in range(ABLATION_EPOCHS):
            train_one_epoch(encoder, dataset, tr_idx, optimizer, cfg_base, device)
            if (epoch + 1) % 5 == 0:
                val_res = evaluate_protonet(encoder, dataset, tr_idx, val_idx, device)
                if val_res['macro_f1'] > best_val_f1:
                    best_val_f1 = val_res['macro_f1']
                    best_state = {k: v.clone() for k, v in encoder.state_dict().items()}

        if best_state:
            encoder.load_state_dict(best_state)

        test_res = evaluate_protonet(encoder, dataset, tr_idx, te_idx, device)
        fold_f1s.append(test_res['macro_f1'])
        print(f"    Fold {fold_idx+1}: F1={test_res['macro_f1']:.3f}")

    return {
        'macro_f1_mean': float(np.mean(fold_f1s)),
        'macro_f1_std': float(np.std(fold_f1s)),
        'fold_f1s': [float(f) for f in fold_f1s],
    }


# Baseline config using best settings from previous steps
shuttle_base_cfg = get_config('shuttle_ablation', **{
    'ablation.feature_layer': BEST_STEP1,
    'ablation.encoder': BEST_STEP2,
    'ablation.use_pretrained': BEST_STEP3_PRETRAINED,
})

step6_results = {}

print(f"{'='*50}")
print(f"Step 6a — Skeleton only (34 nodes)")
step6_results['skeleton_only'] = run_shuttle_ablation(
    use_shuttle=False,
    best_feature_layer=BEST_STEP1,
    best_encoder_name=BEST_STEP2,
    best_use_pretrained=BEST_STEP3_PRETRAINED,
    cfg_base=shuttle_base_cfg,
    device=device,
)

print(f"\n{'='*50}")
print(f"Step 6b — Skeleton + Shuttle (35 nodes)")
step6_results['skeleton_plus_shuttle'] = run_shuttle_ablation(
    use_shuttle=True,
    best_feature_layer=BEST_STEP1,
    best_encoder_name=BEST_STEP2,
    best_use_pretrained=BEST_STEP3_PRETRAINED,
    cfg_base=shuttle_base_cfg,
    device=device,
)

# Summary
print(f"\n{'='*60}")
print(f"Step 6 Summary — Shuttlecock Input Ablation")
print(f"  Fixed: feature={BEST_STEP1}, encoder={BEST_STEP2}, pretrained={BEST_STEP3_PRETRAINED}")
print(f"{'Variant':<25} {'Macro-F1':>10} {'± Std':>8}")
print("-" * 45)
for name, res in step6_results.items():
    marker = " ←" if res['macro_f1_mean'] == max(r['macro_f1_mean'] for r in step6_results.values()) else ""
    print(f"  {name:<23} {res['macro_f1_mean']:>10.3f} {res['macro_f1_std']:>8.3f}{marker}")

delta = step6_results['skeleton_plus_shuttle']['macro_f1_mean'] - step6_results['skeleton_only']['macro_f1_mean']
print(f"\n  Shuttle Δ: {delta:+.3f} ({'benefit' if delta > 0 else 'no benefit'})")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 4))
names = list(step6_results.keys())
means = [step6_results[n]['macro_f1_mean'] for n in names]
stds  = [step6_results[n]['macro_f1_std']  for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=5, color=['#4C72B0', '#DD8452'])
ax.set_ylabel('Macro-F1')
ax.set_title('Step 6 — Shuttlecock Input Ablation')
ax.set_ylim(0, max(means) + 0.15)
ax.axhline(0.2, color='gray', linestyle='--', linewidth=0.8, label='Random baseline')
ax.legend()
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ablation_shuttle.png', dpi=150)
plt.show()

---
## Step 6 — Shuttlecock Trajectory: Does Ball Position Help?

**Why this matters:** So far we only use player skeleton data (34 body joints). But badminton strategy is partly defined by *where the shuttle goes* — a player might be in an attacking posture but if the shuttle is behind them, the actual strategy is defensive recovery.

We add the shuttlecock as a **35th virtual node** in the skeleton graph, connected to both players' dominant wrists. This lets the ST-GCN learn joint-shuttle spatial relationships directly.

| Variant | Graph nodes | Description |
|---------|:----------:|-------------|
| **skeleton_only** | 34 | Current approach — 17 joints per player |
| **skeleton + shuttle** | 35 | Add shuttle (x, y, visibility) as node 34, connected to wrist joints |

**What we expect:** If shuttle tracking is accurate, adding it should help disambiguate strategies like *attack* vs *defend* where body pose is similar but shuttle position differs. However, noisy tracking (low visibility frames) could add noise instead.

Shuttle data source: `datasets_preprocessing/finebadminton_shuttles/{rally_id}.npy`, shape `(T, 3)` — per-frame [x, y, visibility].

**Fixed settings:** All best from Steps 1–5.